In [2]:
'''***********DO NOT RUN, THESE ARE THE TABLES************
USERS
+---------+---------+---------------------+-------+
| user_id |  name   |        email        | region|
+---------+---------+---------------------+-------+
|   1     | Alice   | alice@example.com   | East  |
|   2     | Bob     | bob@example.com     | West  |
|   3     | Charlie | charlie@example.com | East  |
|   4     | David   | david@example.com   | West  |
|   5     | Emma    | emma@example.com    | East  |
+---------+---------+---------------------+-------+



ORDERS
+----------+---------+------------+--------+
| order_id | user_id |  product   | amount |
+----------+---------+------------+--------+
|   101    |    1    | Phone      | 700    |
|   102    |    2    | Console    | 400    |
|   103    |    1    | Headphones | 150    |
|   104    |    3    | Keyboard   | 120    |
|   105    |    2    | Tablet     | 500    |
|   106    |    3    | Monitor    | 250    |
|   107    |    3    | Mouse      | 50     |
|   108    |    1    | Smartwatch | 300    |
+----------+---------+------------+--------+


PRODUCTS
+-----------+---------------+---------+
| product_id| product_name  |  price  |
+-----------+---------------+---------+
|    1      | Phone         | 700     |
|    2      | Console       | 400     |
|    3      | Headphones    | 150     |
|    4      | Keyboard      | 120     |
|    5      | Tablet        | 500     |
|    6      | Monitor       | 250     |
|    7      | Mouse         | 50      |
|    8      | Smartwatch    | 300     |
+-----------+---------------+---------+   '''

'***********DO NOT RUN, THESE ARE THE TABLES************\nUSERS\n+---------+---------+---------------------+-------+\n| user_id |  name   |        email        | region|\n+---------+---------+---------------------+-------+\n|   1     | Alice   | alice@example.com   | East  |\n|   2     | Bob     | bob@example.com     | West  |\n|   3     | Charlie | charlie@example.com | East  |\n|   4     | David   | david@example.com   | West  |\n|   5     | Emma    | emma@example.com    | East  |\n+---------+---------+---------------------+-------+\n\n\n\nORDERS\n+----------+---------+------------+--------+\n| order_id | user_id |  product   | amount |\n+----------+---------+------------+--------+\n|   101    |    1    | Phone      | 700    |\n|   102    |    2    | Console    | 400    |\n|   103    |    1    | Headphones | 150    |\n|   104    |    3    | Keyboard   | 120    |\n|   105    |    2    | Tablet     | 500    |\n|   106    |    3    | Monitor    | 250    |\n|   107    |    3    | Mouse  

In [3]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 6.6 MB/s eta 0:00:00


In [4]:
# redis_crud.py
# Requires: pip install redis

from typing import Optional, Dict, Any, List
import redis

# =========================
# Redis Connection
# =========================
import redis

"""Basic connection example.
"""

import redis

r = redis.Redis(
    host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com',
    port=17263,
    decode_responses=True,
    username="default",
    password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN",
)

success = r.set('foo', 'bar')
# True

result = r.get('foo')
print(result)
# >>> bar



# Test connection
try:
    r.ping()
    print("✅ Connected to Redis!")
except redis.ConnectionError as e:
    print("❌ Redis connection failed:", e)


# -----------------
# Helper utilities
# -----------------
def _exists(r: redis.Redis, key: str) -> bool:
    return bool(r.exists(key))


def _to_int(x: Any) -> Optional[int]:
    try:
        return int(x)
    except Exception:
        return None


# =========== USER CRUD ===========
def add_user(user_id: int, name: str, email: str, region: str = "East") -> bool:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "email": email, "region": region})
    pipe.sadd("users", uid)
    pipe.execute()
    return True


def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    data = r.hgetall(key)
    if not data:
        return None
    return {"user_id": uid, **data}


def update_user_name(user_id: int, name: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_user_email(user_id: int, email: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "email", email))


def delete_user(user_id: int) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False

    order_set_key = f"orders_by_user:{uid}"
    order_ids = list(r.smembers(order_set_key))

    pipe = r.pipeline()
    for oid in order_ids:
        pipe.delete(f"order:{uid}:{oid}")
    pipe.delete(order_set_key)
    pipe.srem("users", uid)
    pipe.delete(key)
    pipe.execute()
    return True


def get_all_users() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("users")])
    out: List[Dict[str, Any]] = []
    for uid in ids:
        u = get_user(uid)
        if u:
            out.append(u)
    return out


# =========== ORDER CRUD ===========
def add_order(user_id: int, order_id: int, product: str, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    if uid is None or oid is None:
        raise ValueError("user_id and order_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return False

    key = f"order:{uid}:{oid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"product": product, "amount": str(amount)})
    pipe.sadd(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


def get_orders_by_user(user_id: int) -> List[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return []

    set_key = f"orders_by_user:{uid}"
    ids = sorted([int(x) for x in r.smembers(set_key)])
    out: List[Dict[str, Any]] = []
    for oid in ids:
        key = f"order:{uid}:{oid}"
        data = r.hgetall(key)
        if data:
            out.append(
                {
                    "user_id": uid,
                    "order_id": oid,
                    "product": data.get("product"),
                    "amount": float(data.get("amount", "0") or "0"),
                }
            )
    return out


def update_order_product(user_id: int, order_id: int, product: str) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "product", product))


def update_order_amount(user_id: int, order_id: int, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "amount", str(amount)))


def delete_order(user_id: int, order_id: int) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


# =========== QUERY HELPERS ===========
def get_orders_for_region(region: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        if u.get("region") == region:
            out.extend(get_orders_by_user(u["user_id"]))
    return out


def total_sales_for_user(user_id: int) -> float:
    return sum(o["amount"] for o in get_orders_by_user(user_id))


def orders_amount_gt(threshold: float) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        for o in get_orders_by_user(u["user_id"]):
            if o["amount"] > threshold:
                out.append(o)
    return out


# =========== PRODUCT CRUD ===========
def add_product(product_id: int, name: str, price: float) -> bool:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "price": str(price)})
    pipe.sadd("products", pid)
    pipe.execute()
    return True


def get_product(product_id: int) -> Optional[Dict[str, Any]]:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    data = r.hgetall(key)
    if not data:
        return None

    return {
        "product_id": pid,
        "name": data.get("name"),
        "price": float(data.get("price", "0") or "0"),
    }


def get_all_products() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("products")])
    out: List[Dict[str, Any]] = []
    for pid in ids:
        p = get_product(pid)
        if p:
            out.append(p)
    return out


def update_product_name(product_id: int, name: str) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_product_price(product_id: int, price: float) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "price", str(price)))


def delete_product(product_id: int) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem("products", pid)
    pipe.execute()
    return True


bar
✅ Connected to Redis!


In [5]:
# export_redis_to_json.py
import json
import redis

r = redis.Redis(host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com', port=17263, db=0, password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN", decode_responses=True)

out = {}
for key in r.scan_iter(match='*', count=1000):
    t = r.type(key)
    if t == 'string':
        out[key] = r.get(key)
    elif t == 'hash':
        out[key] = r.hgetall(key)
    elif t == 'list':
        out[key] = r.lrange(key, 0, -1)
    elif t == 'set':
        out[key] = list(r.smembers(key))
    elif t == 'zset':
        out[key] = r.zrange(key, 0, -1, withscores=True)
    else:
        out[key] = {"_type": t}
# Save
with open('redis_dump.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)


In [6]:
#ALL 15 USERS
add_user(1, "Alice", "alice@example.com", "East"  )
add_user(2, "Bob", "bob@example.com", "West"  )
add_user(3, "Charlie", "charlie@example.com", "East"  )
add_user(4, "David", "david@example.com", "West"  )
add_user(5, "Emma", "emma@example.com", "East")
add_user(6, "Frank", "frank@example.com", "West")
add_user(7, "Grace", "grace@example.com", "East")
add_user(8, "Heidi", "heidi@example.com", "West")
add_user(9, "Ivan", "ivan@example.com", "East")
add_user(10, "Judy", "judy@example.com", "West")
add_user(11, "Kevin", "kevin@example.com", "East")
add_user(12, "Liam", "liam@example.com", "West")
add_user(13, "Mia", "mia@example.com", "East")
add_user(14, "Noah", "noah@example.com", "West")
add_user(15, "Olivia", "olivia@example.com", "East")

False

In [7]:
#ALL 18 PRODUCTS
add_product(1, "Phone", 700)
add_product(2, "Console", 400)
add_product(3, "Headphones", 150)
add_product(4, "Keyboard", 120)
add_product(5, "Tablet", 500)
add_product(6, "Monitor", 250)
add_product(7, "Mouse", 50)
add_product(8, "Smartwatch", 300)
add_product(9, "Laptop", 1000)
add_product(10, "Camera", 600)
add_product(11, "Speaker", 200)
add_product(12, "Printer", 350)
add_product(13, "Webcam", 80)
add_product(14, "Router", 100)
add_product(15, "External Hard Drive", 120)
add_product(16, "Headset", 70)
add_product(17, "Microphone", 90)
add_product(18, "Drone", 800)

False

In [8]:
#ALL 18 ORDERS
add_order(1, 101, "Phone", 700)
add_order(2, 102, "Console", 400)
add_order(1, 103, "Headphones", 150)
add_order(3, 104, "Keyboard", 120)
add_order(2, 105, "Tablet", 500)
add_order(3, 106, "Monitor", 250)
add_order(3, 107, "Mouse", 50)
add_order(4, 108, "Smartwatch", 300)
add_order(4, 109, "Laptop", 1000)
add_order(5, 110, "Camera", 600)
add_order(5, 111, "Speaker", 200)
add_order(6, 112, "Printer", 350)
add_order(6, 113, "Webcam", 80)
add_order(7, 114, "Router", 100)
add_order(7, 115, "External Hard Drive", 120)
add_order(8, 116, "Headset", 70)
add_order(8, 117, "Microphone", 90)
add_order(9, 118, "Drone", 800)




False

In [9]:
# Task 1: Retrieve user 1 and write down their name and email.
u1 = get_user(1)
print("Task 1 - User 1 name:", u1["name"])
print("Task 1 - User 1 email:", u1["email"])


Task 1 - User 1 name: Alice
Task 1 - User 1 email: new_email@example.com


In [10]:
# Task 2: Retrieve all users and write down how many users the system contains.
users = get_all_users()
print("Task 2 - Number of users:", len(users))


Task 2 - Number of users: 15


In [11]:
# Task 3: Retrieve all orders belonging to user 1 and list the product names.
orders_u1 = get_orders_by_user(1)
print("Task 3 - Product names for user 1 orders:", [o["product"] for o in orders_u1])


Task 3 - Product names for user 1 orders: ['Phone', 'Headphones', 'Smartwatch']


In [12]:
# Task 4: Retrieve all products and write down all product names.
products = get_all_products()
print("Task 4 - All product names:", [p["name"] for p in products])


Task 4 - All product names: ['Phone', 'New Console', 'Headphones', 'Keyboard', 'Tablet', 'Monitor', 'Mouse', 'Smartwatch', 'Laptop', 'Camera', 'Speaker', 'Printer', 'Webcam', 'Router', 'External Hard Drive', 'Headset', 'Microphone', 'Drone']


In [13]:
# Task 5: Retrieve product 3 and write down its product name and price.
p3 = get_product(3)
print("Task 5 - Product 3 name:", p3["name"])
print("Task 5 - Product 3 price:", p3["price"])


Task 5 - Product 3 name: Headphones
Task 5 - Product 3 price: 150.0


In [14]:
# Task 6: Retrieve the total sales amount for user 1 and record the value.
total_u1 = total_sales_for_user(1)
print("Task 6 - Total sales for user 1:", total_u1)


Task 6 - Total sales for user 1: 1150.0


In [15]:
# Task 7: Retrieve all orders whose amount is greater than 500 and list their order IDs.
orders_gt_500 = orders_amount_gt(500)
print("Task 7 - Order IDs with amount > 500:", [o["order_id"] for o in orders_gt_500])


Task 7 - Order IDs with amount > 500: [101, 109, 110, 118]


In [16]:
# Task 8: Retrieve all orders placed by users in the East region and list all order IDs.
east_orders = get_orders_for_region("East")
print("Task 8 - East region order IDs:", [o["order_id"] for o in east_orders])


Task 8 - East region order IDs: [101, 103, 108, 104, 106, 107, 110, 111, 114, 115, 118]


In [17]:
# Task 9: Update the email of user 1, then retrieve the user and record the updated email.
update_user_email(1, "alice_updated@example.com")
u1_updated = get_user(1)
print("Task 9 - Updated email for user 1:", u1_updated["email"])


Task 9 - Updated email for user 1: alice_updated@example.com


In [18]:
# Task 10: Update the name of product 2, then retrieve the product and write down the updated name.
update_product_name(2, "Updated Product 2")
p2_updated = get_product(2)
print("Task 10 - Updated name for product 2:", p2_updated["name"])


Task 10 - Updated name for product 2: Updated Product 2


In [19]:
# Task 11: Retrieve all users, identify one user from the West region, then retrieve their orders and write down all product names.
users = get_all_users()
west_users = [u for u in users if u.get("region") == "West"]
if west_users:
    west_user = west_users[0]
    west_orders = get_orders_by_user(west_user["user_id"])
    print("Task 11 - West user chosen:", west_user)
    print("Task 11 - Product names for that user's orders:", [o["product"] for o in west_orders])
else:
    print("Task 11 - No West-region users found.")


Task 11 - West user chosen: {'user_id': 2, 'name': 'Bob', 'email': 'bob@example.com', 'region': 'West'}
Task 11 - Product names for that user's orders: ['Console', 'Tablet']


In [20]:
# Task 12: Retrieve the orders of any user of your choice, then retrieve matching product information and compare prices.
user_choice = 1
orders_choice = get_orders_by_user(user_choice)
products = get_all_products()
price_by_name = {p["name"]: p["price"] for p in products}

print("Task 12 - Comparing order amounts with product list prices for user", user_choice)
for o in orders_choice:
    list_price = price_by_name.get(o["product"])
    print(f"Order {o['order_id']}: product={o['product']}, order_amount={o['amount']}, list_price={list_price}")


Task 12 - Comparing order amounts with product list prices for user 1
Order 101: product=Phone, order_amount=700.0, list_price=700.0
Order 103: product=Headphones, order_amount=150.0, list_price=150.0
Order 108: product=Smartwatch, order_amount=300.0, list_price=300.0


In [21]:
# Task 13: Retrieve all products and all orders, then determine whether every product name in the orders exists in the product list.
products = get_all_products()
orders_all = orders_amount_gt(0)
product_names = {p["name"] for p in products}
order_product_names = {o["product"] for o in orders_all}

missing = order_product_names - product_names
print("Task 13 - Are there any order product names missing from products list?", bool(missing))
print("Task 13 - Missing product names (if any):", missing)


Task 13 - Are there any order product names missing from products list? True
Task 13 - Missing product names (if any): {'Console'}


In [22]:
# Task 14: Retrieve the total sales of user 1 and user 2 and identify which user has higher total sales.
total_1 = total_sales_for_user(1)
total_2 = total_sales_for_user(2)

if total_1 > total_2:
    higher = 1
elif total_2 > total_1:
    higher = 2
else:
    higher = None

print("Task 14 - Total sales user 1:", total_1)
print("Task 14 - Total sales user 2:", total_2)
print("Task 14 - User with higher total sales:", higher)


Task 14 - Total sales user 1: 1150.0
Task 14 - Total sales user 2: 900.0
Task 14 - User with higher total sales: 1


In [23]:
# Task 15: Increase the price of product 5, retrieve it again, and write down the updated price.
p5 = get_product(5)
if p5 is not None:
    new_price_5 = p5["price"] + 10.0
    update_product_price(5, new_price_5)
    p5_updated = get_product(5)
    print("Task 15 - Updated price for product 5:", p5_updated["price"])
else:
    print("Task 15 - Product 5 not found.")


Task 15 - Updated price for product 5: 510.0


In [24]:
# Task 16: Retrieve all orders above 300, then retrieve the associated users and write down their names.
orders_gt_300 = orders_amount_gt(300)
user_names = set()
for o in orders_gt_300:
    u = get_user(o["user_id"])
    if u:
        user_names.add(u["name"])
print("Task 16 - Users with orders > 300:", sorted(user_names))


Task 16 - Users with orders > 300: ['Alice', 'Bob', 'David', 'Emma', 'Frank', 'Ivan']


In [25]:
# Task 17: Retrieve product 1 and the orders of user 1, then compare the product price to the amounts in those orders.
p1 = get_product(1)
orders_u1 = get_orders_by_user(1)
print("Task 17 - Product 1 price:", p1["price"] if p1 else None)
print("Task 17 - User 1 order amounts vs product price:")
for o in orders_u1:
    print(f"Order {o['order_id']}: amount={o['amount']}, product_price={p1['price'] if p1 else None}")


Task 17 - Product 1 price: 700.0
Task 17 - User 1 order amounts vs product price:
Order 101: amount=700.0, product_price=700.0
Order 103: amount=150.0, product_price=700.0
Order 108: amount=300.0, product_price=700.0


In [26]:
# Task 18: Delete product 7 and then retrieve all products to confirm whether product 7 appears.
deleted = delete_product(7)
products_after = get_all_products()
ids_after = [p["product_id"] for p in products_after]
print("Task 18 - delete_product(7) returned:", deleted)
print("Task 18 - Product IDs after deletion:", ids_after)


Task 18 - delete_product(7) returned: True
Task 18 - Product IDs after deletion: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [27]:
# Task 19: Retrieve all users, select any user, update their name, then retrieve that user again and write down the updated name.
users = get_all_users()
if users:
    chosen = users[0]
    new_name = chosen["name"] + " (Updated)"
    update_user_name(chosen["user_id"], new_name)
    chosen_updated = get_user(chosen["user_id"])
    print("Task 19 - Updated user:", chosen_updated)
else:
    print("Task 19 - No users found.")


Task 19 - Updated user: {'user_id': 1, 'name': 'Alice (Updated)', 'email': 'alice_updated@example.com', 'region': 'East'}


In [28]:
# Task 20: Retrieve all users from the East region, then retrieve total sales for each and write the totals.
east_users = [u for u in get_all_users() if u.get("region") == "East"]
print("Task 20 - Total sales by East-region users:")
for u in east_users:
    total = total_sales_for_user(u["user_id"])
    print(f"User {u['user_id']} ({u['name']}): total_sales={total}")


Task 20 - Total sales by East-region users:
User 1 (Alice (Updated)): total_sales=1150.0
User 3 (Charlie): total_sales=420.0
User 5 (Emma): total_sales=800.0
User 7 (Grace): total_sales=220.0
User 9 (Ivan): total_sales=800.0
User 11 (Kevin): total_sales=0
User 13 (Mia): total_sales=0
User 15 (Olivia): total_sales=0


In [29]:
# Task 21: Retrieve all orders above 0 and all products, then determine whether any order amount differs from the corresponding product price.
orders_all = orders_amount_gt(0)
products = get_all_products()
price_by_name = {p["name"]: p["price"] for p in products}

mismatches = []
for o in orders_all:
    list_price = price_by_name.get(o["product"])
    if list_price is not None and o["amount"] != list_price:
        mismatches.append((o["order_id"], o["product"], o["amount"], list_price))

print("Task 21 - Any mismatches between order amounts and product prices?", bool(mismatches))
print("Task 21 - Mismatches:", mismatches)


Task 21 - Any mismatches between order amounts and product prices? True
Task 21 - Mismatches: [(105, 'Tablet', 500.0, 510.0)]


In [30]:
# Task 22: Retrieve all orders above 0, choose any order, then retrieve the associated user and write down their name.
orders_all = orders_amount_gt(0)
if orders_all:
    chosen_order = orders_all[0]
    user_for_order = get_user(chosen_order["user_id"])
    print("Task 22 - Chosen order:", chosen_order)
    print("Task 22 - User for chosen order:", user_for_order["name"] if user_for_order else None)
else:
    print("Task 22 - No orders found.")


Task 22 - Chosen order: {'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}
Task 22 - User for chosen order: Alice (Updated)


In [31]:
# Task 23: Retrieve all orders above 0 and determine manually which three have the highest amounts.
orders_all = orders_amount_gt(0)
orders_sorted = sorted(orders_all, key=lambda o: o["amount"], reverse=True)
top3 = orders_sorted[:3]
print("Task 23 - Top 3 orders by amount:", top3)


Task 23 - Top 3 orders by amount: [{'user_id': 4, 'order_id': 109, 'product': 'Laptop', 'amount': 1000.0}, {'user_id': 9, 'order_id': 118, 'product': 'Drone', 'amount': 800.0}, {'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}]


In [32]:
# Task 24: Retrieve all products, identify the most expensive product, then determine whether any order uses that product.
products = get_all_products()
if products:
    most_expensive = max(products, key=lambda p: p["price"])
    orders_all = orders_amount_gt(0)
    used_in_orders = any(o["product"] == most_expensive["name"] for o in orders_all)
    print("Task 24 - Most expensive product:", most_expensive)
    print("Task 24 - Is it used in any order?", used_in_orders)
else:
    print("Task 24 - No products found.")


Task 24 - Most expensive product: {'product_id': 9, 'name': 'Laptop', 'price': 1000.0}
Task 24 - Is it used in any order? True


In [33]:
# Task 25: Retrieve two different products and compare their prices.
products = get_all_products()
if len(products) >= 2:
    p_a, p_b = products[0], products[1]
    print("Task 25 - Product A:", p_a)
    print("Task 25 - Product B:", p_b)
    if p_a["price"] > p_b["price"]:
        print("Task 25 - Product A is more expensive.")
    elif p_b["price"] > p_a["price"]:
        print("Task 25 - Product B is more expensive.")
    else:
        print("Task 25 - Both products have the same price.")
else:
    print("Task 25 - Not enough products to compare.")


Task 25 - Product A: {'product_id': 1, 'name': 'Phone', 'price': 700.0}
Task 25 - Product B: {'product_id': 2, 'name': 'Updated Product 2', 'price': 400.0}
Task 25 - Product A is more expensive.


In [34]:
# Task 26: Add a new order for any user, then retrieve that user’s orders and write down the newly added order.
user_id = 1
existing_orders = get_orders_by_user(user_id)
new_order_id = max([o["order_id"] for o in existing_orders], default=0) + 1
new_order_amount = 123.45
new_order_product = "NewOrderProduct"

added = add_order(user_id, new_order_id, new_order_product, new_order_amount)
orders_after = get_orders_by_user(user_id)
new_order = [o for o in orders_after if o["order_id"] == new_order_id]

print("Task 26 - Added new order?", added)
print("Task 26 - Newly added order:", new_order)


Task 26 - Added new order? True
Task 26 - Newly added order: [{'user_id': 1, 'order_id': 109, 'product': 'NewOrderProduct', 'amount': 123.45}]


In [35]:
# Task 27: Retrieve user 3, delete user 3, then retrieve all users and orders and write down what changed.
u3_before = get_user(3)
orders_3_before = get_orders_by_user(3)
deleted = delete_user(3)
u3_after = get_user(3)
orders_3_after = get_orders_by_user(3)

print("Task 27 - User 3 before deletion:", u3_before)
print("Task 27 - Orders for user 3 before deletion:", orders_3_before)
print("Task 27 - delete_user(3) returned:", deleted)
print("Task 27 - User 3 after deletion:", u3_after)
print("Task 27 - Orders for user 3 after deletion:", orders_3_after)


Task 27 - User 3 before deletion: {'user_id': 3, 'name': 'Charlie', 'email': 'charlie@example.com', 'region': 'East'}
Task 27 - Orders for user 3 before deletion: [{'user_id': 3, 'order_id': 104, 'product': 'Keyboard', 'amount': 120.0}, {'user_id': 3, 'order_id': 106, 'product': 'Monitor', 'amount': 250.0}, {'user_id': 3, 'order_id': 107, 'product': 'Mouse', 'amount': 50.0}]
Task 27 - delete_user(3) returned: True
Task 27 - User 3 after deletion: None
Task 27 - Orders for user 3 after deletion: []


In [36]:
# Task 28: Retrieve all East-region orders and compare the product names with the product table.
east_orders = get_orders_for_region("East")
products = get_all_products()
product_names = {p["name"] for p in products}
order_names = {o["product"] for o in east_orders}

missing_in_products = order_names - product_names
print("Task 28 - Product names in East orders:", order_names)
print("Task 28 - Names from East orders missing in_products:", missing_in_products)


Task 28 - Product names in East orders: {'Headphones', 'Drone', 'Router', 'Speaker', 'External Hard Drive', 'NewOrderProduct', 'Smartwatch', 'Phone', 'Camera'}
Task 28 - Names from East orders missing in_products: {'NewOrderProduct'}


In [37]:
# Task 29: Update the price of product 1, add a new order using that product, then retrieve user 1’s orders and write down the new order amount.
p1_before = get_product(1)
if p1_before is not None:
    new_price_1 = p1_before["price"] + 50.0
    update_product_price(1, new_price_1)
    p1_after = get_product(1)

    orders_u1 = get_orders_by_user(1)
    new_order_id = max([o["order_id"] for o in orders_u1], default=0) + 1
    add_order(1, new_order_id, p1_after["name"], p1_after["price"])

    orders_u1_after = get_orders_by_user(1)
    new_order = [o for o in orders_u1_after if o["order_id"] == new_order_id]
    print("Task 29 - Product 1 updated price:", p1_after["price"])
    print("Task 29 - Newly added order for user 1:", new_order)
else:
    print("Task 29 - Product 1 not found.")


Task 29 - Product 1 updated price: 750.0
Task 29 - Newly added order for user 1: [{'user_id': 1, 'order_id': 110, 'product': 'Phone', 'amount': 750.0}]


In [38]:
# Task 30: Lower the price of product 6 and retrieve the product again to confirm the updated price.
p6_before = get_product(6)
if p6_before is not None:
    new_price_6 = max(0.0, p6_before["price"] - 10.0)
    update_product_price(6, new_price_6)
    p6_after = get_product(6)
    print("Task 30 - Product 6 price before:", p6_before["price"])
    print("Task 30 - Product 6 price after:", p6_after["price"])
else:
    print("Task 30 - Product 6 not found.")


Task 30 - Product 6 price before: 250.0
Task 30 - Product 6 price after: 240.0


In [39]:
# Task 31: Retrieve all users and determine which user has the longest name.
users = get_all_users()
if users:
    longest_user = max(users, key=lambda u: len(u["name"] or ""))
    print("Task 31 - User with longest name:", longest_user)
else:
    print("Task 31 - No users found.")


Task 31 - User with longest name: {'user_id': 1, 'name': 'Alice (Updated)', 'email': 'alice_updated@example.com', 'region': 'East'}


In [40]:
# Task 32: Retrieve all orders above 0 and identify which product name appears most frequently.
orders_all = orders_amount_gt(0)
freq = {}
for o in orders_all:
    freq[o["product"]] = freq.get(o["product"], 0) + 1

if freq:
    most_common_product = max(freq.items(), key=lambda kv: kv[1])
    print("Task 32 - Order counts by product:", freq)
    print("Task 32 - Most frequently ordered product:", most_common_product)
else:
    print("Task 32 - No orders found.")


Task 32 - Order counts by product: {'Phone': 2, 'Headphones': 1, 'Smartwatch': 2, 'NewOrderProduct': 1, 'Console': 1, 'Tablet': 1, 'Laptop': 1, 'Camera': 1, 'Speaker': 1, 'Printer': 1, 'Webcam': 1, 'Router': 1, 'External Hard Drive': 1, 'Headset': 1, 'Microphone': 1, 'Drone': 1}
Task 32 - Most frequently ordered product: ('Phone', 2)


In [41]:
# Task 33: Update both the name and email of user 2, then retrieve user 2 and write down both updated fields.
update_user_name(2, "User 2 Updated")
update_user_email(2, "user2_updated@example.com")
u2_updated = get_user(2)
print("Task 33 - Updated user 2:", u2_updated)


Task 33 - Updated user 2: {'user_id': 2, 'name': 'User 2 Updated', 'email': 'user2_updated@example.com', 'region': 'West'}


In [42]:
# Task 34: Retrieve all orders of user 1 and compute the average order amount manually from the printed values.
orders_u1 = get_orders_by_user(1)
amounts = [o["amount"] for o in orders_u1]
print("Task 34 - User 1 order amounts:", amounts)
if amounts:
    avg_amount = sum(amounts) / len(amounts)
else:
    avg_amount = 0.0
print("Task 34 - Average order amount for user 1:", avg_amount)


Task 34 - User 1 order amounts: [700.0, 150.0, 300.0, 123.45, 750.0]
Task 34 - Average order amount for user 1: 404.69


In [43]:
# Task 35: Retrieve all products and determine which one has the smallest price.
products = get_all_products()
if products:
    cheapest = min(products, key=lambda p: p["price"])
    print("Task 35 - Cheapest product:", cheapest)
else:
    print("Task 35 - No products found.")


Task 35 - Cheapest product: {'product_id': 16, 'name': 'Headset', 'price': 70.0}


In [44]:
# Task 36: Retrieve all orders above 300 and compare them to the product table to identify which orders correspond to expensive products.
orders_gt_300 = orders_amount_gt(300)
products = get_all_products()
price_by_name = {p["name"]: p["price"] for p in products}

expensive_orders = []
for o in orders_gt_300:
    list_price = price_by_name.get(o["product"])
    if list_price is not None and list_price > 300:
        expensive_orders.append((o, list_price))

print("Task 36 - Orders > 300 that correspond to expensive products (price > 300):", expensive_orders)


Task 36 - Orders > 300 that correspond to expensive products (price > 300): [({'user_id': 1, 'order_id': 101, 'product': 'Phone', 'amount': 700.0}, 750.0), ({'user_id': 1, 'order_id': 110, 'product': 'Phone', 'amount': 750.0}, 750.0), ({'user_id': 2, 'order_id': 105, 'product': 'Tablet', 'amount': 500.0}, 510.0), ({'user_id': 4, 'order_id': 109, 'product': 'Laptop', 'amount': 1000.0}, 1000.0), ({'user_id': 5, 'order_id': 110, 'product': 'Camera', 'amount': 600.0}, 600.0), ({'user_id': 6, 'order_id': 112, 'product': 'Printer', 'amount': 350.0}, 350.0), ({'user_id': 9, 'order_id': 118, 'product': 'Drone', 'amount': 800.0}, 800.0)]


In [45]:
# Task 37: Add a new product, retrieve all products, and determine where the new product fits in the price ranking.
products_before = get_all_products()
existing_ids = [p["product_id"] for p in products_before]
new_product_id = max(existing_ids, default=0) + 1
new_product_price = 275.0

add_product(new_product_id, f"Product{new_product_id}", new_product_price)
products_after = get_all_products()
products_sorted = sorted(products_after, key=lambda p: p["price"])

rank_index = None
for idx, p in enumerate(products_sorted):
    if p["product_id"] == new_product_id:
        rank_index = idx
        break

print("Task 37 - New product id:", new_product_id)
print("Task 37 - New product price ranking index (0-based):", rank_index)
print("Task 37 - Products sorted by price:", products_sorted)


Task 37 - New product id: 19
Task 37 - New product price ranking index (0-based): 9
Task 37 - Products sorted by price: [{'product_id': 16, 'name': 'Headset', 'price': 70.0}, {'product_id': 13, 'name': 'Webcam', 'price': 80.0}, {'product_id': 17, 'name': 'Microphone', 'price': 90.0}, {'product_id': 14, 'name': 'Router', 'price': 100.0}, {'product_id': 4, 'name': 'Keyboard', 'price': 120.0}, {'product_id': 15, 'name': 'External Hard Drive', 'price': 120.0}, {'product_id': 3, 'name': 'Headphones', 'price': 150.0}, {'product_id': 11, 'name': 'Speaker', 'price': 200.0}, {'product_id': 6, 'name': 'Monitor', 'price': 240.0}, {'product_id': 19, 'name': 'Product19', 'price': 275.0}, {'product_id': 8, 'name': 'Smartwatch', 'price': 300.0}, {'product_id': 12, 'name': 'Printer', 'price': 350.0}, {'product_id': 2, 'name': 'Updated Product 2', 'price': 400.0}, {'product_id': 5, 'name': 'Tablet', 'price': 510.0}, {'product_id': 10, 'name': 'Camera', 'price': 600.0}, {'product_id': 1, 'name': 'Phone'

In [46]:
# Task 38: Retrieve all users and all orders, then determine which user has the greatest number of orders.
users = get_all_users()
orders_all = orders_amount_gt(0)

order_counts = {u["user_id"]: 0 for u in users}
for o in orders_all:
    if o["user_id"] in order_counts:
        order_counts[o["user_id"]] += 1

if order_counts:
    best_uid, best_count = max(order_counts.items(), key=lambda kv: kv[1])
    best_user = get_user(best_uid)
    print("Task 38 - Order counts per user:", order_counts)
    print("Task 38 - User with most orders:", best_user, "with", best_count, "orders")
else:
    print("Task 38 - No orders or users to compare.")


Task 38 - Order counts per user: {1: 5, 2: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 1, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}
Task 38 - User with most orders: {'user_id': 1, 'name': 'Alice (Updated)', 'email': 'alice_updated@example.com', 'region': 'East'} with 5 orders


In [47]:
# Task 39: Update the prices of products 1 and 2, then retrieve all products and confirm the updated prices.
p1 = get_product(1)
p2 = get_product(2)

if p1 is not None:
    update_product_price(1, p1["price"] + 5.0)
if p2 is not None:
    update_product_price(2, p2["price"] + 5.0)

products_after = get_all_products()
print("Task 39 - Products after updating prices of 1 and 2:", products_after)


Task 39 - Products after updating prices of 1 and 2: [{'product_id': 1, 'name': 'Phone', 'price': 755.0}, {'product_id': 2, 'name': 'Updated Product 2', 'price': 405.0}, {'product_id': 3, 'name': 'Headphones', 'price': 150.0}, {'product_id': 4, 'name': 'Keyboard', 'price': 120.0}, {'product_id': 5, 'name': 'Tablet', 'price': 510.0}, {'product_id': 6, 'name': 'Monitor', 'price': 240.0}, {'product_id': 8, 'name': 'Smartwatch', 'price': 300.0}, {'product_id': 9, 'name': 'Laptop', 'price': 1000.0}, {'product_id': 10, 'name': 'Camera', 'price': 600.0}, {'product_id': 11, 'name': 'Speaker', 'price': 200.0}, {'product_id': 12, 'name': 'Printer', 'price': 350.0}, {'product_id': 13, 'name': 'Webcam', 'price': 80.0}, {'product_id': 14, 'name': 'Router', 'price': 100.0}, {'product_id': 15, 'name': 'External Hard Drive', 'price': 120.0}, {'product_id': 16, 'name': 'Headset', 'price': 70.0}, {'product_id': 17, 'name': 'Microphone', 'price': 90.0}, {'product_id': 18, 'name': 'Drone', 'price': 800.0}

In [48]:
# Task 40: Retrieve all orders above 0 and all products, then determine whether any product has never been ordered.
orders_all = orders_amount_gt(0)
products = get_all_products()

ordered_names = {o["product"] for o in orders_all}
never_ordered = [p for p in products if p["name"] not in ordered_names]

print("Task 40 - Products never ordered:", never_ordered)


Task 40 - Products never ordered: [{'product_id': 2, 'name': 'Updated Product 2', 'price': 405.0}, {'product_id': 4, 'name': 'Keyboard', 'price': 120.0}, {'product_id': 6, 'name': 'Monitor', 'price': 240.0}, {'product_id': 19, 'name': 'Product19', 'price': 275.0}]


In [49]:
# Task 41: Retrieve all users and write down the name of the first user in the list.
users = sorted(get_all_users(), key=lambda u: u["user_id"])
if users:
    print("Task 41 - First user name:", users[0]["name"])
else:
    print("Task 41 - No users found.")


Task 41 - First user name: Alice (Updated)


In [50]:
# Task 42: Retrieve all users and write down the email of the second user in the list.
users = sorted(get_all_users(), key=lambda u: u["user_id"])
if len(users) >= 2:
    print("Task 42 - Second user email:", users[1]["email"])
else:
    print("Task 42 - Fewer than two users found.")


Task 42 - Second user email: user2_updated@example.com


In [51]:
# Task 43: Retrieve all orders of user 1 and write down the product name of the first order.
orders_u1 = get_orders_by_user(1)
if orders_u1:
    print("Task 43 - Product name of first order for user 1:", orders_u1[0]["product"])
else:
    print("Task 43 - User 1 has no orders.")


Task 43 - Product name of first order for user 1: Phone


In [52]:
# Task 44: Retrieve all orders of user 1 and write down the amount of the last order.
orders_u1 = get_orders_by_user(1)
if orders_u1:
    print("Task 44 - Amount of last order for user 1:", orders_u1[-1]["amount"])
else:
    print("Task 44 - User 1 has no orders.")


Task 44 - Amount of last order for user 1: 750.0


In [53]:
# Task 45: Retrieve all products and write down the name of the third product.
products = get_all_products()
if len(products) >= 3:
    print("Task 45 - Name of third product:", products[2]["name"])
else:
    print("Task 45 - Fewer than three products found.")


Task 45 - Name of third product: Headphones


In [54]:
# Task 46: Retrieve user 1 and write down this user’s email only.
u1 = get_user(1)
print("Task 46 - Email of user 1:", u1["email"] if u1 else None)


Task 46 - Email of user 1: alice_updated@example.com


In [55]:
# Task 47: Retrieve product 1 and write down this product’s price only.
p1 = get_product(1)
print("Task 47 - Price of product 1:", p1["price"] if p1 else None)


Task 47 - Price of product 1: 755.0


In [56]:
# Task 48: Retrieve all East-region orders and write down the user ID of the first returned order.
east_orders = get_orders_for_region("East")
if east_orders:
    print("Task 48 - User ID of first East-region order:", east_orders[0]["user_id"])
else:
    print("Task 48 - No East-region orders found.")


Task 48 - User ID of first East-region order: 1


In [57]:
# Task 49: Retrieve all orders above 500 and write down the product name of the first returned order.
orders_gt_500 = orders_amount_gt(500)
if orders_gt_500:
    print("Task 49 - Product name of first order with amount > 500:", orders_gt_500[0]["product"])
else:
    print("Task 49 - No orders with amount > 500 found.")


Task 49 - Product name of first order with amount > 500: Phone


In [58]:
# Task 50: Retrieve all users and all orders for user 1 and write down a pair consisting of the name of the first user and the product of the first order of user 1.
users = sorted(get_all_users(), key=lambda u: u["user_id"])
orders_u1 = get_orders_by_user(1)

first_user_name = users[0]["name"] if users else None
first_order_product = orders_u1[0]["product"] if orders_u1 else None

print("Task 50 - (first user name, first product for user 1):", (first_user_name, first_order_product))


Task 50 - (first user name, first product for user 1): ('Alice (Updated)', 'Phone')
